In [4]:
!pip install opencv-python tensorflow
import os
import cv2
import numpy as np
import pandas as pd
import logging
import tensorflow as tf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 36.2 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LSTM, TimeDistributed,
    GlobalAveragePooling2D, BatchNormalization, Bidirectional
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

In [ ]:
TRAIN_REAL_DIR = "video/train/real"
TRAIN_FAKE_DIR = "video/train/fake"
TEST_REAL_DIR = "video/test/real"
TEST_FAKE_DIR = "video/test/fake"

In [ ]:
IMG_SIZE = 112
NUM_FRAMES = 20
BATCH_SIZE = 4
EPOCHS = 20

FRAME_CACHE = {}

logging.basicConfig(level=logging.INFO)

In [ ]:
def load_data():
    data = []

    def add(folder, label):
        if not os.path.exists(folder):
            return
        for f in os.listdir(folder):
            if f.endswith((".mp4", ".avi")):
                data.append({
                    "video_path": os.path.join(folder, f),
                    "label": label
                })

    add(TRAIN_REAL_DIR, 1)
    add(TRAIN_FAKE_DIR, 0)
    add(TEST_REAL_DIR, 1)
    add(TEST_FAKE_DIR, 0)

    df = pd.DataFrame(data)
    df["is_train"] = df["video_path"].str.contains("train")

    logging.info(f"Total videos: {len(df)}")
    return df

In [ ]:
def extract_frames(video_path, num_frames):
    if video_path in FRAME_CACHE:
        return FRAME_CACHE[video_path]

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        return None

    indices = np.linspace(0, total - 1, num_frames, dtype=int)
    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()

        if not ret:
            frame = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))

        frames.append(frame)

    cap.release()

    frames = np.array(frames)
    FRAME_CACHE[video_path] = frames
    return frames

In [ ]:
def data_generator(df, batch_size):
    while True:
        df = df.sample(frac=1)

        for i in range(0, len(df), batch_size):
            batch = df.iloc[i:i+batch_size]

            X, y = [], []

            for _, row in batch.iterrows():
                frames = extract_frames(row["video_path"], NUM_FRAMES)
                if frames is None:
                    continue

                frames = preprocess_input(frames.astype(np.float32))

                X.append(frames)
                y.append(row["label"])

            if len(X) > 0 :
                yield np.array(X), np.array(y)

In [ ]:
def build_model():
    base_model = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    base_model.trainable = False

    inputs = Input(shape=(NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3))

    x = TimeDistributed(base_model)(inputs)
    x = TimeDistributed(GlobalAveragePooling2D())(x)

    # Improved temporal modeling
    x = Bidirectional(LSTM(128, return_sequences=True))(x)
    x = Bidirectional(LSTM(64))(x)

    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)

    x = Dense(64, activation="relu")(x)
    x = Dropout(0.3)(x)

    outputs = Dense(1, activation="sigmoid")(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4, clipnorm=1.0),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC()]
    )

    return model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = "/content/drive/MyDrive/video"

TRAIN_REAL_DIR = f"{BASE_PATH}/train/real"
TRAIN_FAKE_DIR = f"{BASE_PATH}/train/fake"
TEST_REAL_DIR  = f"{BASE_PATH}/test/real"
TEST_FAKE_DIR  = f"{BASE_PATH}/test/fake"

In [ ]:
if __name__ == "__main__":
    df = load_data()

    df_train = df[df["is_train"]]
    df_test = df[~df["is_train"]]

    df_train, df_val = train_test_split(df_train, test_size=0.2, random_state=42)

    train_gen = data_generator(df_train, BATCH_SIZE)
    val_gen = data_generator(df_val, BATCH_SIZE)

    model = build_model()
    model.summary()

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    lr_scheduler = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )

    checkpoint = ModelCheckpoint(
        f"{BASE_PATH}/cnn_lstm_model.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )

    model.fit(
        train_gen,
        steps_per_epoch=max(1,len(df_train) // BATCH_SIZE),
        validation_data=val_gen,
        validation_steps=max(1,len(df_val) // BATCH_SIZE),
        epochs=EPOCHS,
        callbacks=[early_stop, lr_scheduler, checkpoint]
    )

    # ======================================
    # EVALUATION
    # ======================================
    X_test, y_test = [], []

    for _, row in df_test.iterrows():
        frames = extract_frames(row["video_path"], NUM_FRAMES)
        if frames is None:
            continue

        frames = preprocess_input(frames.astype(np.float32))
        X_test.append(frames)
        y_test.append(row["label"])

    X_test = np.array(X_test)
    y_test = np.array(y_test)

    preds = model.predict(X_test)
    y_pred = (preds > 0.5).astype(int)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    # Save both .keras and .h5 models
    model.save(f"{BASE_PATH}/cnn_lstm_model.keras")
    print("Model saved as cnn_lstm_model.keras")
    model.save(f"{BASE_PATH}/cnn_lstm_model.h5")
    print("Model saved as cnn_lstm_model.h5")

    print("\nROC-AUC:", roc_auc_score(y_test, preds))